# Demand: RieszNet with a continuous log-price treatment

Adapted from `RieszNet_BHP.ipynb`: use the same `Learner` and `avg_small_diff` moment for a continuous treatment. Train once per supplied train/test fold, with an internal training-only validation split. Independently retain a forecasting checkpoint and an elasticity checkpoint from the same run.

- Treatment: **`log_price_ratio_msrp`**, already supplied as log(selling price / MSRP); do not log it again.
- Outcome: **log(sales)**, using the natural logarithm, not log1p.
- Confounders: all remaining schema features except `log_selling_price`. In particular, keep `log_price_ratio_gfk_street_price` and `log_price_ratio_competitor_asp` as requested. `sample_weight` is export metadata, not a schema feature; this uses the repository's unweighted loss.
- Fit categorical OHE, numeric imputation, and numeric scaling on the internal training split only. Leave treatment and log outcome unscaled so derivatives retain log/log units.

The two learned heads are **g(X)** (predicted log sales) and **a(X)** (the Riesz representer). The representer is not itself elasticity. Predict row-level elasticity as the derivative of g with respect to log price ratio, approximated using BHP's central difference (step 0.01). Also report the Riesz debiased average elasticity, mean[derivative g + a × (log sales − g)], on the held-out test fold. Elasticity and its debiasing correction use g and a from the elasticity checkpoint together.

Report MAE in log space and MAE/WMAPE in sales units after exponentiating g. Raw sales and raw treatment are never model inputs. WMAPE = 100 × sum(abs(sales − predicted sales)) / sum(abs(sales)). Exponentiating g estimates a geometric conditional sales level; no retransformation correction is applied. There are no ground-truth elasticity labels, so elasticity is summarized, not scored for accuracy. Interpretation holds the chosen confounders fixed, including the retained price ratios.

In [1]:
from pathlib import Path
import json
import random
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from utils.NN_avgmom_sim import Learner
from utils.riesznet import RieszNet
from utils.moments import avg_small_diff

DATA_DIR = Path("data/demand-feature-csv")
RESULT_DIR = Path("results/demand/RieszNet")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
FOLDS = (1, 2, 3, 4)
TREATMENT = "log_price_ratio_msrp"
TARGET = "sales"
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Same BHP architecture; weaker RR-head decay and validation sales-MAE selection.
n_hidden = 100
drop_prob = 0.0
train_opt = dict(
    n_epochs=40, bs=128, learner_lr=0.0002, learner_l2=1e-2, riesz_l2=1e-4,
    earlystop_rounds=7, earlystop_delta=2e-4, earlystop_metric="sales_mae",
    scheduler_patience=5, secondary_checkpoint_metric="regression",
    target_reg=1, riesz_weight=0.03, optimizer="adam",
)
print("Device:", DEVICE)

Device: cuda


## Load and prepare each fold

Rows with missing/nonfinite log treatment or nonpositive/nonfinite sales cannot be used for the log/log task; count and remove them explicitly. Missing confounders are imputed. Unseen test categories are handled by OHE without fitting on test data.

In [2]:
def read_fold(fold):
    schema = json.loads((DATA_DIR / f"fold_{fold}_schema.json").read_text())
    confounders = [c for c in schema["features"]
                   if c not in {TREATMENT, "log_selling_price"}]
    categorical = [c for c in confounders if c in schema["categories"]]
    numeric = [c for c in confounders if c not in categorical]
    frames, dropped = {}, {}
    for split in ("train", "test"):
        frame = pd.read_csv(
            DATA_DIR / f"fold_{fold}_{split}.csv",
            dtype={c: "str" for c in categorical},
        )
        frame.index.name = "csv_row"  # zero-based data row, preserved after filtering
        valid = (np.isfinite(frame[TREATMENT])
                 & np.isfinite(frame[TARGET]) & (frame[TARGET] > 0))
        dropped[split] = int((~valid).sum())
        frame = frame.loc[valid].copy()
        frame[numeric] = frame[numeric].replace([np.inf, -np.inf], np.nan)
        frames[split] = frame
    if min(len(f) for f in frames.values()) < 2:
        raise ValueError(f"Fold {fold}: insufficient valid rows")
    return frames["train"], frames["test"], confounders, categorical, numeric, dropped


def make_preprocessor(categorical, numeric):
    return ColumnTransformer([
        ("categorical", make_pipeline(
            SimpleImputer(strategy="constant", fill_value="__missing__"),
            OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32),
        ), categorical),
        ("numeric", make_pipeline(
            SimpleImputer(strategy="median", keep_empty_features=True),
            StandardScaler(),
        ), numeric),
    ], sparse_threshold=0)


def make_xy(frame, preprocessor, confounders):
    # The moment function differentiates column 0: it must be log treatment.
    W = preprocessor.transform(frame[confounders]).astype(np.float32)
    X = np.column_stack([frame[TREATMENT].to_numpy(dtype=np.float32), W])
    y = np.log(frame[TARGET].to_numpy(dtype=np.float64)).astype(np.float32)
    if not np.isfinite(X).all() or not np.isfinite(y).all():
        raise ValueError("Nonfinite values remain after preprocessing")
    return X, y


def demand_validation_metrics(Xval, yval, output):
    # yval and head 0 are log sales; inverse-transform only to score predictions.
    log_y = yval.reshape(-1).to(torch.float64)
    g = output[:, 0].to(torch.float64)
    a = output[:, 1].to(torch.float64)
    t = Xval[:, 0].to(torch.float64)
    sales = torch.exp(log_y)
    errors = torch.abs(sales - torch.exp(g))
    return {
        "sales_mae": errors.mean().item(),
        "sales_wmape_pct": (100 * errors.sum() / sales.abs().sum()).item(),
        "rr_mean": a.mean().item(),                         # reference: 0
        "rr_t_moment": (a * t).mean().item(),              # reference: 1
        "rr_t2_moment_error": ((a * t.square()).mean() - 2 * t.mean()).item(),
        "rr_rms": a.square().mean().sqrt().item(),
        "rr_std": a.std(unbiased=False).item(),
    }



def predict_demand_checkpoints(forecast_net, elasticity_net, X, device, batch_size=1024):
    forecast_net.eval()
    elasticity_net.eval()
    forecasts, elasticity_outputs, elasticities = [], [], []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            xb = torch.as_tensor(X[start:start + batch_size], device=device)
            forecasts.append(forecast_net(xb).cpu().numpy())
            elasticity_outputs.append(elasticity_net(xb).cpu().numpy())
            elasticities.append(avg_small_diff(
                xb, lambda x: elasticity_net(x)[:, [0]], device
            ).cpu().numpy().ravel())
    forecast = np.concatenate(forecasts)
    elasticity_output = np.concatenate(elasticity_outputs)
    return {
        "g_log_sales": forecast[:, 0],
        "riesz_representer": forecast[:, 1],
        "g_log_sales_elasticity": elasticity_output[:, 0],
        "riesz_representer_elasticity": elasticity_output[:, 1],
        "predicted_elasticity": np.concatenate(elasticities),
    }


## Train once per fold and evaluate on its test file

Use the same seeded 80/20 split of each train file. The supplied test file is used only for evaluation. Keep two independently selected checkpoints from **one training run**:

- **Forecast:** lowest validation sales MAE after exponentiating g, with the configured minimum improvement. Use this checkpoint for sales predictions, MAE, WMAPE, and reported forecast log MAE.
- **Elasticity:** lowest validation log-sales MSE across epochs actually run. Use this checkpoint for the derivative and for both g and a in the debiased elasticity score. Log MSE is a proxy for regression quality, not proof of elasticity accuracy; inspect the RR diagnostics recorded at this checkpoint.

The architecture is unchanged. Training remains capped at **40 epochs**, batch size **256**, learning rate **0.005**, shared/regression L2 **0.01**, RR-head L2 **0.0001**, and targeted regularization **1**. The Riesz loss weight is now **0.03**. The training objective still uses squared error on log sales plus the existing Riesz and corrected-regression losses.

Scheduling and early stopping still monitor validation sales MAE: scheduler patience **5**, early-stopping patience **7**, and minimum improvement **0.0002 sales units**. The elasticity checkpoint does not extend training: it is the best log-MSE checkpoint among epochs evaluated before sales-based stopping or the 40-epoch cap.

Print/save training losses, validation losses, sales MAE/WMAPE, learning rate, and RR diagnostics at every epoch. RR reference identities are E[a]=0, E[aT]=1, and E[aT²]−2E[T]=0. Training losses are sample-weighted batch averages without an extra training pass; validation uses the fixed model at epoch end.

Save both selected epochs and validation diagnostic sets in `fold_*_checkpoint_metrics.csv` and in the fold summary. Prediction columns `g_log_sales` and `riesz_representer` come from the forecasting checkpoint; `g_log_sales_elasticity` and `riesz_representer_elasticity` come from the elasticity checkpoint. `predicted_elasticity` and `debiased_elasticity_score` use only the elasticity checkpoint. All history/selection epochs are 1-based.


In [3]:
fold_metrics = []
all_predictions = []

for fold in FOLDS:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    train, test, confounders, categorical, numeric, dropped = read_fold(fold)
    fit_frame, val_frame = train_test_split(train, test_size=0.2, random_state=SEED)
    preprocessor = make_preprocessor(categorical, numeric)
    preprocessor.fit(fit_frame[confounders])
    X_train, y_train = make_xy(fit_frame, preprocessor, confounders)
    X_val, y_val = make_xy(val_frame, preprocessor, confounders)
    X_test, y_test = make_xy(test, preprocessor, confounders)

    print(f"Fold {fold}: fit={len(y_train)}, validation={len(y_val)}, "
          f"test={len(y_test)}, inputs={X_train.shape[1]}, dropped={dropped}")
    # Same continuous-treatment learner as utils.NN_avgmom_sim.est_avgmom_NN.
    model = RieszNet(Learner(X_train.shape[1], n_hidden, drop_prob, degree=0),
                     avg_small_diff)
    model.fit(X_train, y_train, Xval=X_val, yval=y_val,
              model_dir=str(RESULT_DIR), device=DEVICE, verbose=1,
              validation_metrics_fn=demand_validation_metrics, **train_opt)
    validation_history = pd.DataFrame(model.validation_history_)
    validation_history.insert(0, "fold", fold)
    validation_history.to_csv(RESULT_DIR / f"fold_{fold}_validation_history.csv", index=False)
    print(f"Forecast epoch {model.best_epoch_} (1-based):", model.best_validation_metrics_)
    print(f"Elasticity epoch {model.secondary_best_epoch_} (1-based):",
          model.secondary_best_validation_metrics_)
    pd.DataFrame([
        dict(role="forecast", epoch=model.best_epoch_, criterion="sales_mae",
             **model.best_validation_metrics_),
        dict(role="elasticity", epoch=model.secondary_best_epoch_, criterion="regression",
             **model.secondary_best_validation_metrics_),
    ]).to_csv(RESULT_DIR / f"fold_{fold}_checkpoint_metrics.csv", index=False)
    forecast_net = model.get_model("earlystop")
    elasticity_net = model.get_model("secondary")

    test_predictions = predict_demand_checkpoints(forecast_net, elasticity_net, X_test, DEVICE)
    g_log = test_predictions["g_log_sales"]
    g_log_elasticity = test_predictions["g_log_sales_elasticity"]
    riesz_elasticity = test_predictions["riesz_representer_elasticity"]
    elasticity = test_predictions["predicted_elasticity"]
    predicted_sales = np.exp(g_log.astype(np.float64))
    if not np.isfinite(predicted_sales).all():
        raise ValueError(f"Fold {fold}: nonfinite sales predictions")
    actual_sales = test[TARGET].to_numpy(dtype=np.float64)
    absolute_error = np.abs(actual_sales - predicted_sales)
    dr_score = elasticity + riesz_elasticity * (y_test - g_log_elasticity)

    metrics = dict(
        fold=fold, n_fit=len(y_train), n_validation=len(y_val), n_test=len(y_test),
        dropped_train=dropped["train"], dropped_test=dropped["test"],
        epochs=model.n_epochs, forecast_epoch=model.best_epoch_,
        elasticity_epoch=model.secondary_best_epoch_,
        log_MAE=float(np.mean(np.abs(y_test - g_log))),
        MAE=float(absolute_error.mean()),
        WMAPE_pct=float(100 * absolute_error.sum() / np.abs(actual_sales).sum()),
        mean_elasticity=float(elasticity.mean()),
        median_elasticity=float(np.median(elasticity)),
        debiased_avg_elasticity=float(dr_score.mean()),
    )
    metrics.update({f"val_forecast_{key}": value
                    for key, value in model.best_validation_metrics_.items()})
    metrics.update({f"val_elasticity_{key}": value
                    for key, value in model.secondary_best_validation_metrics_.items()})
    fold_metrics.append(metrics)
    predictions = test[["sku", "account", "country_code"]].copy()
    predictions.insert(0, "fold", fold)
    predictions["log_price_ratio_msrp"] = X_test[:, 0]
    predictions["log_sales"] = y_test
    for name, values in test_predictions.items():
        predictions[name] = values
    predictions["debiased_elasticity_score"] = dr_score
    predictions["sales"] = actual_sales
    predictions["predicted_sales"] = predicted_sales
    predictions.to_csv(RESULT_DIR / f"fold_{fold}_test_predictions.csv")
    all_predictions.append(predictions.reset_index())
    pd.DataFrame(fold_metrics).to_csv(RESULT_DIR / "fold_metrics.csv", index=False)
    print(f"  MAE={metrics['MAE']:.3f}, WMAPE={metrics['WMAPE_pct']:.2f}%, "
          f"mean elasticity={metrics['mean_elasticity']:.3f}, "
          f"debiased average={metrics['debiased_avg_elasticity']:.3f}")
    model.tempdir.cleanup()
    del model, forecast_net, elasticity_net, X_train, X_val, X_test

Fold 1: fit=139578, validation=34895, test=10985, inputs=413, dropped={'train': 330, 'test': 24}
Epoch #0
Training losses (batch averages): 0.9553619248948103 -1.9734832734073675 0.9552882015299425
Training objective: 1.8514456320851607
Validation losses: 0.75574136 -19.573933 0.7555453
Early-stop metric (sales_mae): 24.87355911813852
Validation diagnostics: {'sales_mae': 24.87355911813852, 'sales_wmape_pct': 50.250413973365916, 'rr_mean': 0.08583113252988499, 'rr_t_moment': 0.761002890915034, 'rr_t2_moment_error': 0.36117746392631883, 'rr_rms': 2.6104630631350965, 'rr_std': 2.609051632429169}
Learning rate: 0.0002
Epoch #1
Training losses (batch averages): 0.7330974357716665 -47.021175383920216 0.7328501903612263
Training objective: 0.05531239624711066
Validation losses: 0.7383981 -53.72259 0.73497784
Early-stop metric (sales_mae): 28.02971264563621
Validation diagnostics: {'sales_mae': 28.02971264563621, 'sales_wmape_pct': 56.6265831643927, 'rr_mean': 0.013161802569865702, 'rr_t_mome

In [5]:
import pandas as pd

# Show all columns (replace 100 with a higher number if you have even more columns)
pd.set_option("display.max_columns", None)

# Optional: Ensure column content isn't truncated if text is long
pd.set_option("display.max_colwidth", None)

# Optional: If you have many rows and want to see more rows vertically as well
pd.set_option("display.max_rows", 100)

# Now display your dataframe normally
display(pooled_metrics)
metrics_df = pd.DataFrame(fold_metrics).set_index("fold")
predictions_df = pd.concat(all_predictions, ignore_index=True)
display(metrics_df)

# Pool held-out observations; do not average fold WMAPEs with equal weights.
errors = np.abs(predictions_df["sales"] - predictions_df["predicted_sales"])
pooled_metrics = pd.DataFrame([{
    "n_test": len(predictions_df),
    "log_MAE": np.mean(np.abs(predictions_df["log_sales"] - predictions_df["g_log_sales"])),
    "MAE": errors.mean(),
    "WMAPE_pct": 100 * errors.sum() / predictions_df["sales"].abs().sum(),
    "mean_elasticity": predictions_df["predicted_elasticity"].mean(),
    "median_elasticity": predictions_df["predicted_elasticity"].median(),
    "debiased_avg_elasticity": predictions_df["debiased_elasticity_score"].mean(),
}])
pooled_metrics.to_csv(RESULT_DIR / "pooled_metrics.csv", index=False)
display(pooled_metrics)
display(predictions_df.head())
print("Saved evaluation outputs to", RESULT_DIR.resolve())

,n_test,log_MAE,MAE,WMAPE_pct,mean_elasticity,median_elasticity,debiased_avg_elasticity
0,46733,0.749529,30.234429,63.133762,-1.064737,-0.94806,-1.754014


,n_fit,n_validation,n_test,dropped_train,dropped_test,epochs,forecast_epoch,elasticity_epoch,log_MAE,MAE,WMAPE_pct,mean_elasticity,median_elasticity,debiased_avg_elasticity,val_forecast_regression,val_forecast_riesz_loss,val_forecast_corrected_regression,val_forecast_joint,val_forecast_sales_mae,val_forecast_sales_wmape_pct,val_forecast_rr_mean,val_forecast_rr_t_moment,val_forecast_rr_t2_moment_error,val_forecast_rr_rms,val_forecast_rr_std,val_elasticity_regression,val_elasticity_riesz_loss,val_elasticity_corrected_regression,val_elasticity_joint,val_elasticity_sales_mae,val_elasticity_sales_wmape_pct,val_elasticity_rr_mean,val_elasticity_rr_t_moment,val_elasticity_rr_t2_moment_error,val_elasticity_rr_rms,val_elasticity_rr_std
fold,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,139578,34895,10985,330,24,19,11,11,0.710770,21.639232,52.885940,-1.028095,-0.908583,0.176458,0.684313,-201.497055,0.684337,-4.676261,23.143325,46.754936,-0.035370,0.919398,0.101294,10.864081,10.864023,0.684313,-201.497055,0.684337,-4.676261,23.143325,46.754936,-0.035370,0.919398,0.101294,10.864081,10.864023
2,137876,34470,11860,338,23,14,6,6,0.752626,29.857039,59.676541,-0.894321,-0.843535,-2.399627,0.664655,-78.476608,0.664747,-1.024896,22.833997,46.249798,-0.902502,0.913381,-0.053253,8.388696,8.340007,0.664655,-78.476608,0.664747,-1.024896,22.833997,46.249798,-0.902502,0.913381,-0.053253,8.388696,8.340007
3,134989,33748,12068,325,26,16,8,8,0.732549,34.553323,60.460599,-1.542805,-1.331675,-2.276914,0.666297,-93.983490,0.666483,-1.486725,21.693468,43.708885,-0.242166,0.945573,-0.145008,9.010900,9.007645,0.666297,-93.983490,0.666483,-1.486725,21.693468,43.708885,-0.242166,0.945573,-0.145008,9.010900,9.007645
4,132611,33153,11820,314,30,13,5,9,0.799779,34.191594,79.951540,-0.781683,-0.744370,-2.366445,0.669224,-62.444702,0.669464,-0.534653,22.950031,46.829274,0.493575,1.174298,0.171870,7.701421,7.685589,0.668401,-124.901070,0.668273,-2.410359,23.327659,47.599819,0.456275,0.996085,-0.090191,9.946766,9.936295


,n_test,log_MAE,MAE,WMAPE_pct,mean_elasticity,median_elasticity,debiased_avg_elasticity
0,46733,0.749529,30.234429,63.133762,-1.064737,-0.94806,-1.754014


,csv_row,fold,sku,account,country_code,log_price_ratio_msrp,log_sales,g_log_sales,riesz_representer,g_log_sales_elasticity,riesz_representer_elasticity,predicted_elasticity,debiased_elasticity_score,sales,predicted_sales
0,0,1,1F3Y2A,X-KOM,PL,1.452734,1.098612,1.316799,0.199491,1.316799,0.199491,-0.926304,-0.969830,3.0,3.731456
1,1,1,1F3Y2A,KOMPUTRONIK,PL,1.452734,3.178054,3.880848,8.156816,3.880848,8.156816,-0.577831,-6.310398,24.0,48.465316
2,2,1,1F3Y2A,MEDIAEXPERT,PL,1.452734,3.583519,4.723743,8.679458,4.723743,8.679458,-0.361276,-10.257801,36.0,112.588881
3,3,1,1F3Y2A,EURO,PL,1.452734,1.609438,3.950638,7.302274,3.950638,7.302274,-0.539517,-17.635605,5.0,51.968527
4,4,1,1F3Y2A,MEDIAMARKT,PL,1.452734,3.332205,4.782334,8.466838,4.782334,8.466838,-0.353265,-12.631278,28.0,119.382703


Saved evaluation outputs to C:\Users\chahuara\Projects\RieszLearning\results\demand\RieszNet
